In [ ]:
import os
import cv2
import numpy as np

processed_path = "/content/drive/MyDrive/processed_data"

X = []
y = []

# label map
label_map = {"HC":0, "PD":1}

# traverse train and test folders
for split in ["train", "test"]:
    split_path = os.path.join(processed_path, split)

    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        if not os.path.isdir(cls_path):
            continue

        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            img = cv2.imread(img_path)

            if img is None:
                print("Skipped invalid image:", img_path)
                continue

            img = cv2.resize(img, (224,224))
            X.append(img)
            y.append(label_map.get(cls, -1))  # HC=0, PD=1

X = np.array(X)
y = np.array(y)

print("Total loaded images:", len(X))
print("Unique labels in y:", np.unique(y))

Total loaded images: 2000
Unique labels in y: [0 1]


In [ ]:
# -----------------------------
# 1️⃣ Install timm & import libraries
# -----------------------------
!pip install timm

# Keras imports
import numpy as np
import os
import cv2
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0, MobileNetV2, DenseNet121, InceptionV3, Xception
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.models import Model

# PyTorch imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm

# -----------------------------
# 2️⃣ Mount Drive & prepare paths
# -----------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

processed_path = "/content/drive/MyDrive/processed_data"
save_path = "/content/drive/MyDrive/Feature extract"
os.makedirs(save_path, exist_ok=True)

# -----------------------------
# 3️⃣ Load processed images
# -----------------------------
X = []
y = []

label_map = {"HC":0, "PD":1}

for split in ["train","test"]:
    split_path = os.path.join(processed_path, split)
    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        if not os.path.isdir(cls_path):
            continue
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            img = cv2.imread(img_path)
            if img is None:
                print("Skipped:", img_path)
                continue
            img = cv2.resize(img, (224,224))
            X.append(img)
            y.append(label_map.get(cls, -1))

X = np.array(X)
y = np.array(y)
print("Total images loaded:", len(X))
print("Unique labels:", np.unique(y))

# -----------------------------
# 4️⃣ Feature extraction for Keras models
# -----------------------------
keras_models = ["VGG16","ResNet50","EfficientNetB0","MobileNetV2","DenseNet121","InceptionV3","Xception"]

def extract_features_keras(model_name, X_images):
    if model_name=="VGG16":
        model = VGG16(weights='imagenet', include_top=False, pooling='avg')
        X_pp = vgg16_preprocess(X_images)
    elif model_name=="ResNet50":
        model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
        X_pp = resnet_preprocess(X_images)
    elif model_name=="EfficientNetB0":
        model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')
        X_pp = efficientnet_preprocess(X_images)
    elif model_name=="MobileNetV2":
        model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
        X_pp = mobilenet_preprocess(X_images)
    elif model_name=="DenseNet121":
        model = DenseNet121(weights='imagenet', include_top=False, pooling='avg')
        X_pp = densenet_preprocess(X_images)
    elif model_name=="InceptionV3":
        model = InceptionV3(weights='imagenet', include_top=False, pooling='avg')
        X_pp = inception_preprocess(X_images)
    elif model_name=="Xception":
        model = Xception(weights='imagenet', include_top=False, pooling='avg')
        X_pp = xception_preprocess(X_images)
    else:
        return None
    feats = model.predict(X_pp, batch_size=32, verbose=1)
    return feats

# Run Keras models
for model_name in keras_models:
    print("Extracting features for:", model_name)
    feats = extract_features_keras(model_name, X)
    if feats is not None:
        np.save(os.path.join(save_path,f"{model_name}_features.npy"), feats)
        np.save(os.path.join(save_path,f"{model_name}_labels.npy"), y)
        print(f"Saved features for {model_name}")

# -----------------------------
# 5️⃣ PyTorch Dataset for AlexNet, ViT, CustomCNN
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class SpectrogramDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.images = []
        self.labels = []
        self.transform = transform
        for split in ["train","test"]:
            split_path = os.path.join(folder_path, split)
            for cls in os.listdir(split_path):
                cls_path = os.path.join(split_path, cls)
                if not os.path.isdir(cls_path):
                    continue
                for img_name in os.listdir(cls_path):
                    img_path = os.path.join(cls_path, img_name)
                    try:
                        img = Image.open(img_path).convert('RGB')
                        self.images.append(img)
                        self.labels.append(label_map.get(cls,-1))
                    except:
                        print("Skipped:", img_path)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])
dataset = SpectrogramDataset(processed_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

# -----------------------------
# 6️⃣ PyTorch Models
# -----------------------------
# AlexNet
alexnet_model = torch.hub.load('pytorch/vision:v0.13.1', 'alexnet', pretrained=True)
alexnet_model.classifier = nn.Identity()
alexnet_model.eval().to(device)

# ViT (timm)
vit_model = timm.create_model('vit_base_patch16_224', pretrained=True)
vit_model.head = nn.Identity()
vit_model.eval().to(device)

# Custom CNN
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN,self).__init__()
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.fc = nn.Linear(64*56*56,512)
    def forward(self,x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc(x))
        return x

custom_model = CustomCNN().to(device)
custom_model.eval()

# -----------------------------
# 7️⃣ Feature Extraction Function (PyTorch)
# -----------------------------
def extract_features_torch(model, dataloader):
    features_list = []
    with torch.no_grad():
        for imgs, _ in dataloader:
            imgs = imgs.to(device)
            feats = model(imgs)
            feats = feats.cpu().numpy()
            features_list.append(feats)
    return np.vstack(features_list)

# -----------------------------
# 8️⃣ Extract & Save PyTorch Model Features
# -----------------------------
pytorch_models = {"AlexNet":alexnet_model, "ViT":vit_model, "CustomCNN":custom_model}

for name, model in pytorch_models.items():
    print("Extracting features for:", name)
    feats = extract_features_torch(model, dataloader)
    np.save(os.path.join(save_path,f"{name}_features.npy"), feats)
    np.save(os.path.join(save_path,f"{name}_labels.npy"), np.array(dataset.labels))
    print(f"Saved {name} features successfully!")


Mounted at /content/drive
Total images loaded: 2000
Unique labels: [0 1]
Extracting features for: VGG16
63/63 ━━━━━━━━━━━━━━━━━━━━ 1235s 20s/step
Saved features for VGG16
Extracting features for: ResNet50
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 377s 6s/step
Saved features for ResNet50
Extracting features for: EfficientNetB0
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 151s 2s/step
Saved features for EfficientNetB0
Extracting features for: MobileNetV2


/tmp/ipython-input-2154842738.py:83: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step
Saved features for MobileNetV2
Extracting features for: DenseNet121
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 364s 6s/step
Saved features for DenseNet121
Extracting features for: InceptionV3
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 255s 4s/step
Saved features for InceptionV3
Extracting features for: Xception
83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 472s 7s/step
Saved features for Xception
Using device: cpu
Downloading: "https://github.com/pytorch/vision/zipball/v0.13.1" to /root/.cache/torch/hub/v0.13.1.zip


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 156MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Extracting features for: AlexNet
Saved AlexNet features successfully!
Extracting features for: ViT
Saved ViT features successfully!
Extracting features for: CustomCNN
Saved CustomCNN features successfully!


In [ ]:
import numpy as np
import os
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# === YOUR DRIVE PATH ===
base = "/content/drive/MyDrive/Feature extract"

# === All 10 feature extractors ===
feature_files = {
    "AlexNet": ("AlexNet_features.npy", "AlexNet_labels.npy"),
    "VGG16": ("VGG16_features.npy", "VGG16_labels.npy"),
    "ResNet50": ("ResNet50_features.npy", "ResNet50_labels.npy"),
    "DenseNet121": ("DenseNet121_features.npy", "DenseNet121_labels.npy"),
    "MobileNetV2": ("MobileNetV2_features.npy", "MobileNetV2_labels.npy"),
    "EfficientNetB0": ("EfficientNetB0_features.npy", "EfficientNetB0_labels.npy"),
    "Xception": ("Xception_features.npy", "Xception_labels.npy"),
    "InceptionV3": ("InceptionV3_features.npy", "InceptionV3_labels.npy"),
    "ViT": ("ViT_features.npy", "ViT_labels.npy"),
    "CustomCNN": ("CustomCNN_features.npy", "CustomCNN_labels.npy")
}

# === 9 Popular ML Classifiers ===
ml_models = {
    "SVM": SVC(kernel="rbf"),
    "Random Forest": RandomForestClassifier(n_estimators=150),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Naive Bayes": GaussianNB(),
    "AdaBoost": AdaBoostClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Extra Trees": ExtraTreesClassifier()
}

# === Result storage ===
results = []

print("🔥 Starting ML Classification for all 10 models...\n")

for model_name, (feature_f, label_f) in feature_files.items():
    feature_path = f"{base}/{feature_f}"
    label_path = f"{base}/{label_f}"

    if not os.path.exists(feature_path):
        print(f"❌ Missing file: {feature_path}")
        continue

    print(f"\n=========================")
    print(f"📌 Feature Extractor: {model_name}")
    print(f"=========================")

    # Load data
    X = np.load(feature_path)
    y = np.load(label_path).reshape(-1)

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    best_acc = 0
    best_model_name = None

    # Apply ML classifiers
    for ml_name, ml_clf in ml_models.items():
        # Fit
        ml_clf.fit(X_train, y_train)
        # Predict
        y_pred = ml_clf.predict(X_test)
        # Accuracy
        acc = accuracy_score(y_test, y_pred)

        print(f"   📊 {ml_name} Accuracy = {acc:.4f}")

        results.append({
            "Feature Extractor": model_name,
            "Classifier": ml_name,
            "Accuracy": acc
        })

        # Update best
        if acc > best_acc:
            best_acc = acc
            best_model_name = ml_name

    print(f"✅ Best Classifier for {model_name}: {best_model_name} ({best_acc:.4f})")

# === Convert to DataFrame for Excel ===
df = pd.DataFrame(results)

# Pivot table for thesis/slide ready format
summary_df = df.pivot(index='Feature Extractor', columns='Classifier', values='Accuracy').reset_index()

# Add Best classifier & Accuracy column
best_acc_list = []
best_model_list = []

for idx, row in summary_df.iterrows():
    best_model = row[1:].idxmax()  # skip 'Feature Extractor'
    best_acc = row[best_model]
    best_acc_list.append(best_acc)
    best_model_list.append(best_model)

summary_df['Best'] = best_model_list
summary_df['Best Accuracy'] = best_acc_list

# === Save Excel ===
save_path = f"{base}/Full_Summary_Table.xlsx"
summary_df.to_excel(save_path, index=False)
print(f"\n✅ Summary Table saved at: {save_path}")


🔥 Starting ML Classification for all 10 models...


📌 Feature Extractor: AlexNet
   📊 SVM Accuracy = 0.9275
   📊 Random Forest Accuracy = 0.8875
   📊 KNN Accuracy = 0.9200
   📊 Logistic Regression Accuracy = 0.9350
   📊 Decision Tree Accuracy = 0.7650
   📊 Naive Bayes Accuracy = 0.6875
   📊 AdaBoost Accuracy = 0.8275
   📊 Gradient Boosting Accuracy = 0.9125
   📊 Extra Trees Accuracy = 0.8725
✅ Best Classifier for AlexNet: Logistic Regression (0.9350)

📌 Feature Extractor: VGG16
   📊 SVM Accuracy = 0.9550
   📊 Random Forest Accuracy = 0.9250
   📊 KNN Accuracy = 0.9300
   📊 Logistic Regression Accuracy = 0.9625
   📊 Decision Tree Accuracy = 0.8625
   📊 Naive Bayes Accuracy = 0.8450
   📊 AdaBoost Accuracy = 0.9325
   📊 Gradient Boosting Accuracy = 0.9475
   📊 Extra Trees Accuracy = 0.9525
✅ Best Classifier for VGG16: Logistic Regression (0.9625)

📌 Feature Extractor: ResNet50
   📊 SVM Accuracy = 0.9425
   📊 Random Forest Accuracy = 0.9150
   📊 KNN Accuracy = 0.9250
   📊 Logistic Regressio